# 📘 学习注释版：Silver CRM Sales

**Input：** `workspace.bronze.crm_sales_details`  
**Output：** `workspace.silver.crm_sales`

主要学习：
- 日期有效性检查
- `yyyyMMdd` → Date
- 错误/缺失价格重新计算
- 字段重命名


#Initialization

## 🧰 学习说明：导入 PySpark 函数/类型

这里仅准备后续清洗需要的 API，例如：
- `col()`：引用列
- `trim()`：去前后空格
- `StringType`：判断字符串类型
- `F.when()`：类似 SQL `CASE WHEN`

这一 Cell 不改变数据。


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# Read Bronze table

## 📖 学习说明：读取 Bronze Table

**Input：** `workspace.bronze.crm_sales_details`  
**Process：** `spark.table()` 把 Catalog 中的 Table 读取成 DataFrame  
**Output：** 变量 `df`

后续所有 Silver 清洗都在这个 DataFrame 上进行。


In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

#Silver Transformations

##Trimming

## ✂️ 学习说明：批量 Trim 字符串

遍历 DataFrame 所有字段：

```text
如果字段类型 = String
→ trim()
→ 去掉前后空格
```

真实数据中 `" Jon"` 和 `"Jon"` 在比较/JOIN 时可能被认为不同，所以 Silver 要先清理。


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Cleaning Dates

## 📅 学习说明：销售日期清洗

规则：

```text
日期 = 0 或长度 != 8
→ NULL

有效 8 位日期
→ yyyyMMdd → Date
```

这是非常典型的数据质量处理。


In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

##Sales and Price Corrections

## 🧮 学习说明：修正异常 Price

如果 `price` 为 NULL 或 <= 0：

```text
quantity != 0
→ price = sales / quantity
```

这是根据业务关系反推出缺失/错误值的示例。


In [0]:

df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)


## Renaming Columns

## 🏷️ 学习说明：统一字段名称

把源系统字段名统一成业务更容易理解的名字。

例如：

```text
cst_id        → customer_id
cst_key       → customer_number
prd_nm        → product_name
```

意义：Silver 不只是“清洗值”，也统一数据模型/字段契约。


In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Silver Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.silver.crm_sales`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

## Sanity checks of silver table

## ✅ 学习说明：验证 Silver Sales

重点检查：
- 日期是否变为 Date
- 无效日期是否为 NULL
- price 修正是否正常


### 💡 VS Code 阅读版：Databricks SQL（仅展示，不在本地执行）

```sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10
```

> 原可执行版本仍保留在 `01_可执行注释版`。在 Databricks 中请执行那一版。
